In [1]:
import pandas as pd
import numpy as np
import random
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory
import yfinance as yf
from pathlib import Path
import os
import matplotlib.pyplot as plt


In [2]:
anos = ['2025-12-31',
 '2024-12-31',
 '2023-12-31',
 '2022-12-31',
 '2021-12-31',
 '2020-12-31',
 '2019-12-31',
 '2018-12-31',
 '2017-12-31',
 '2016-12-31',
 '2015-12-31',
 ]

## LER OS RETORNOS, SCORE e SELIC

In [3]:
# Retornos

dfs = {}
for ano in anos:
    ano_usado = ano.split('-')[0]
    nome = f'df_ativos_{ano_usado}.csv'
    caminho = Path('retornos_ativos') / nome
    dfs[ano_usado] = pd.read_csv(caminho).set_index('date')

    

In [4]:
## SELIC para Sigma e excesso
selic_d = pd.read_csv('selic/selic_diario.csv').set_index('date')

In [5]:
## SCORE para score
score_todos_anos = pd.read_csv('score_piotroski/piotroski.csv').set_index('Unnamed: 0').reset_index()

In [6]:
score_todos_anos.rename(columns={'Unnamed: 0': 'date'}, inplace=True)

In [7]:
score_todos_anos.set_index('date', inplace=True)

In [8]:
lista_ativos_finais = score_todos_anos.columns.tolist()
score_todos_anos

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,
2015,0.375,0.625,0.125,0.500,0.125,0.625,0.625,0.375,0.375,0.375,...,0.625,0.750,0.125,0.875,0.125,0.125,0.000,0.375,0.625,0.250
2016,0.625,0.750,0.375,0.750,0.750,0.000,0.500,0.500,0.500,0.625,...,0.375,0.875,0.500,0.375,0.250,1.000,0.000,0.875,0.875,0.500
2017,0.625,0.625,1.000,0.500,0.625,0.625,0.500,0.375,0.375,0.500,...,0.375,0.750,0.625,0.375,0.875,0.750,0.875,0.875,0.375,0.875
2018,0.625,0.875,0.375,0.750,0.500,0.750,0.750,0.125,0.125,0.250,...,0.500,0.750,0.750,0.500,0.875,1.000,0.750,0.625,0.875,0.875
2019,0.375,0.625,0.125,0.375,0.375,0.750,0.625,0.500,0.500,0.375,...,0.375,0.375,0.625,0.500,0.750,0.500,0.500,0.375,1.000,0.500
2020,0.375,0.500,0.250,0.750,0.500,0.625,0.500,0.250,0.250,0.500,...,0.625,0.500,0.625,0.500,0.625,0.875,0.500,0.625,0.750,0.375
2021,0.500,0.750,0.500,0.500,0.750,0.625,0.500,0.375,0.375,0.500,...,0.500,0.125,0.500,0.500,0.875,0.875,0.625,0.375,0.625,0.500
2022,0.375,0.625,0.750,0.375,0.625,0.750,0.750,0.500,0.500,0.625,...,0.500,0.250,0.875,0.625,0.500,0.375,0.375,0.500,0.500,0.625
2023,0.750,0.625,0.625,0.625,0.375,0.375,0.625,0.250,0.250,0.500,...,0.375,0.625,0.875,0.625,0.500,0.625,0.875,0.750,0.875,0.875


## EXCESSO DOS ANOS e SIGMA (MAtriz de covariancia)

##### EXCESSO PARA TODOS

In [9]:
dict_sigma = {}
dict_excesso = {}
for an in anos:
    ano = an.split("-")[0]
    if ano == '2015':
        pass
    else:
        try:
            print(f"=============== \n EXCESSO {ano}\n ============")
            print("Atualização, Ano: ",ano)
            df = dfs[ano]
            # df_f = pd.DataFrame(eval(df))
            df_f = df.copy()
            slc = selic_d[selic_d.index.isin(df_f.index)]
            slc['valor_diario'] = slc['valor_diario']/100

            print(f"Tamanho DF de {ano}: ", len(df_f))
            print(f"Tamanho Selic: ", len(slc['valor_diario']))
            ano = int(ano)
            dict_excesso[ano] = df_f.sub(slc['valor_diario'],axis=0)
            print("tamanho final do Excesso: ",len(dict_excesso[ano]))

            print(f"============\n SIGMA {ano}\n===========")            
            dict_sigma[ano] = dict_excesso[ano].cov()
            print('Tamanho final do SIGMA: ',len(dict_sigma[ano]))

        except Exception as e:
            print(e)
            print("ERror")

 EXCESSO 2025
Atualização, Ano:  2025
Tamanho DF de 2025:  250
Tamanho Selic:  250
tamanho final do Excesso:  250
 SIGMA 2025
Tamanho final do SIGMA:  78
 EXCESSO 2024
Atualização, Ano:  2024
Tamanho DF de 2024:  251
Tamanho Selic:  251
tamanho final do Excesso:  251
 SIGMA 2024
Tamanho final do SIGMA:  78
 EXCESSO 2023
Atualização, Ano:  2023
Tamanho DF de 2023:  246
Tamanho Selic:  246
tamanho final do Excesso:  246
 SIGMA 2023
Tamanho final do SIGMA:  78
 EXCESSO 2022
Atualização, Ano:  2022
Tamanho DF de 2022:  251
Tamanho Selic:  251
tamanho final do Excesso:  251
 SIGMA 2022
Tamanho final do SIGMA:  78
 EXCESSO 2021
Atualização, Ano:  2021
Tamanho DF de 2021:  249
Tamanho Selic:  249
tamanho final do Excesso:  249
 SIGMA 2021
Tamanho final do SIGMA:  78
 EXCESSO 2020
Atualização, Ano:  2020
Tamanho DF de 2020:  246
Tamanho Selic:  246
tamanho final do Excesso:  246
 SIGMA 2020
Tamanho final do SIGMA:  78
 EXCESSO 2019
Atualização, Ano:  2019
Tamanho DF de 2019:  250
Tamanho Selic

## Hiperparâmetros

In [10]:
vb_cardinalidade_max = 10
vb_cardinalidade_min = 10
vb_peso_maximo = 0.20
vb_peso_minimo = 0.02
vb_theta = 0.5

### Fazendo otimização ano a ano e comparando com o proximo ano

In [11]:
anos = ['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']
print(anos)

['2015-12-31', '2016-12-31', '2017-12-31', '2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31', '2022-12-31', '2023-12-31', '2024-12-31', '2025-12-31']


In [12]:
score_usado = score_todos_anos[score_todos_anos.index.isin(['2015'])]
score_usado

,ABEV3,ALOS3,ANIM3,AXIA3,AZZA3,B3SA3,BBAS3,BBDC3,BBDC4,BBSE3,...,TAEE11,TEND3,TOTS3,UGPA3,USIM5,VALE3,VBBR3,VIVT3,WEGE3,YDUQ3
date,,,,,,,,,,,,,,,,,,,,,


In [13]:
from pydoc import text


y = []

carteiras_anuais = {}

melhor_pesos = None
historico = []
carteiras_criadas = []
for a in anos:
    y.append(a.split('-')[0])

print("=-"*48)
print("Anos totais: ",y)
print("=-"*48)

for an in anos:
    ano = an.split("-")[0]
    print("COMEÇANDO ANO NOVO: ",ano)
    #deletar modelo
    if 'model' in locals():
        del model
        print('Modelo Antigo deletado \n Iniciando Novo')
    else:
        print("Nao consta model")
        pass

    print("-"*15)
    print("Começando Modelo do ano: ", ano)
    print("="*6)

    try:

        score_usado = score_todos_anos[score_todos_anos.index.isin([int(ano)])]
        # print(score_usado)
        print("Score Atualizado")
        

        ano_um = int(ano)+1
        # df_usado = f'df_ativos_{ano_um}'
        # retorno_usado = pd.DataFrame(eval(df_usado))
        df_usado = dfs[str(ano_um)]
        retorno_usado = df_usado.copy()
        retorno_usado = retorno_usado[lista_ativos_finais]
        print("Retornos atualizados")

        excesso_usado = dict_excesso[ano_um]
        print("Excessos Atualizados")
        
        sigma_usado = dict_sigma[ano_um]
        print("Sigmas Atualizados")
        
        print("-----")
    except Exception as e:
        print("ERRRRRRRRRRRROR")
        print(e)

    # if df_usado == 'df_ativos_2015':
    #     continue
    # else:
    print("# ------ CRIAÇÃO DO MODELO")
    print("## UTILIZANDO SCORE DO ANO DE: ",ano)
    print("## UTILIZANDO DADOS DE RETORNO DE: ",str(ano_um))
    print("## UTILIZANDO EXCESSO DO ANO DE: ",ano_um)
    print("## UTILIZANDO SIGMAS DO ANO DE: ",ano_um)

    model = pyo.ConcreteModel()

    #---------VARIÁVEIS-----------
    model.nome_ativos = pyo.Set(initialize = lista_ativos_finais)
    model.ativos = pyo.RangeSet(0, len(lista_ativos_finais)-1)
    model.dias = pyo.RangeSet(0, len(retorno_usado)-1)
    model.retornos_ativos = pyo.Param(model.dias, model.ativos, initialize=lambda model,dia, ativo: retorno_usado.iloc[dia, ativo])    
    model.theta = pyo.Param(initialize=vb_theta)
    model.score = pyo.Param(model.ativos, initialize=lambda model,a: score_usado.iloc[0,a])
    model.cardinalidade_valor_max = pyo.Param(initialize=vb_cardinalidade_max)
    model.cardinalidade_valor_min = pyo.Param(initialize=vb_cardinalidade_min)
    model.peso_maximo = pyo.Param(initialize=vb_peso_maximo)
    model.peso_minimo = pyo.Param(initialize=vb_peso_minimo)
    model.x = pyo.Var(model.ativos, bounds=(0,1))
    model.y = pyo.Var(model.ativos, within=pyo.Binary)
    model.excesso = pyo.Param( model.ativos , initialize = lambda model,a: excesso_usado.mean().iloc[a])
    model.sigma = pyo.Param(model.ativos, model.ativos, initialize = lambda model,a,b: sigma_usado.iloc[a,b])
    model.s = pyo.Param(initialize = 2, mutable=True)
    model.r = pyo.Var(within=pyo.NonNegativeReals)
    
    #-------------------------------------- FUNÇÕES

    #=============================
    # Função Objetivo
    #=============================

    def func_objetivo_1(model):
        retorno_esperado = model.theta * sum(
            model.retornos_ativos[dia, a] * model.x[a] for a in model.ativos for dia in model.dias
        )
        # retorno_esperado = model.theta * model.r
        # A dúvida de qual retorno usar (Retorno BRUTO ou Excesso (retorno - selic))

        score_total = (1 - model.theta) * sum(
            model.x[a] * model.score[a] for a in model.ativos
        )
        return retorno_esperado + score_total
    model.obj1 = pyo.Objective(rule=func_objetivo_1, sense=pyo.maximize)

    #=============================
    # RESTRIÇÕES
    #=============================

    def def_r(model):
        return model.r == sum(model.excesso[a]*model.x[a]for a in model.ativos)
    model.const_def_r = pyo.Constraint(rule=def_r)

    ## modelos de Programação de Cone de Segunda Ordem (SOCP) 
    ## e Programação Quadrática com Restrições (QCP)
    def cone(model):
        return model.r**2 >= model.s**2 * sum(model.x[a]*model.sigma[a,b]*model.x[b] for a in model.ativos for b in model.ativos)
    model.constr_cone = pyo.Constraint(rule=cone)

    #REstricao 1 x só ativa se y = 1
    def restr_vinculo_x_y(model, a):
        return model.x[a] <= model.y[a]
    model.const_restr_vinculo_x_y = pyo.Constraint(model.ativos, rule=restr_vinculo_x_y)

    #peso maximo por acao
    def rule_peso_maximo(model, a):
        # return model.x[a] <= 1/model.cardinalidade_valor
        return model.x[a] <= model.peso_maximo
    model.const_peso_maximo = pyo.Constraint(model.ativos, rule=rule_peso_maximo)

    #peso minimo por acao
    def rule_peso_minimo(model, a):
        return model.x[a] >= model.peso_minimo * model.y[a]  # se y=1, então x >= 0.05
    model.const_peso_minimo = pyo.Constraint(model.ativos, rule=rule_peso_minimo)

    #Restrição 2 soma peso 1
    def soma_peso_1(model):
        return sum(model.x[a] for a in model.ativos) == 1
    model.const_soma_peso_1 = pyo.Constraint(rule=soma_peso_1)


    def cardinalidade_min(model):
        return sum(
            model.y[a] for a in model.ativos
            ) >= model.cardinalidade_valor_min
    model.const_cardinalidade_total_min = pyo.Constraint(rule=cardinalidade_min)

    def cardinalidade_max(model):
        return sum(
            model.y[a] for a in model.ativos
            ) <= model.cardinalidade_valor_max
    model.const_cardinalidade_total_max = pyo.Constraint(rule=cardinalidade_max)


    # NOTEBOOOK
    # opt = SolverFactory('cplex', executable='C:\\CPLEX_Studio2211\\cplex\\bin\\x64_win64\\cplex.exe')

    # PC
    opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')

    s_lo = 0.01        # <-- o Sharpe DIÁRIO 
    s_hi = 1   # teto: 2x o melhor ativo individual
    tol  = 0.001
    print("MODEL.S.VALUE => ",model.s.value)
    melhor_pesos = None
    historico = []
    carteiras_criadas = []
    # print(f"Valor a ser batido com S_LO: {s_lo} e S_HI: {s_hi} ... Diferença: {s_hi-s_lo}")
    print(f"Começando o WHILE do ano {ano}")
    while s_hi - s_lo > tol:
        s_a_ser_usado = 0.5 * (s_lo + s_hi)
        # print(s_a_ser_usado)
        model.s.value = s_a_ser_usado
        print("="*18)
        # print("Objetivo: (É o S lá da restrição de sharpe): ",s_a_ser_usado)
        res = opt.solve(model, load_solutions=False, tee=False)
        tc = res.solver.termination_condition
        print(f"Modelo Resolveu....Condição: ",tc)
        if tc == pyo.TerminationCondition.optimal:
            model.solutions.load_from(res)
            melhor_pesos = {a: pyo.value(model.x[a]) for a in model.ativos}
            # print({x:i for x,i in melhor_pesos.items() if i>0})
            s_lo = model.s.value
            print(f"Valor encontrado para o Sharpe: ",s_lo)
            # print(f"Valor FO: ",pyo.value(model.obj1))
            # print(f"Atualização de Valores para próxima rodagem: TC {tc} com S_LO: {s_lo} e S_HI: {s_hi} ... Diferença: {s_hi-s_lo}")
            # print("="*28)
            # for x,i in melhor_pesos.items():
            #     if i>0:

            #         dc = {
            #         'ativos_selecionados':melhor_pesos,
            #         'sharpe':pyo.value(model.s),
            #             }
            historico.append((s_a_ser_usado, 'viavel'))
            # carteiras_criadas.append(dc)
        elif tc in (pyo.TerminationCondition.infeasible,
                    pyo.TerminationCondition.infeasibleOrUnbounded):
            s_hi = s_a_ser_usado
            print(f"Atualização do S_HI : {s_hi}")
            print("="*28)

            historico.append((s_a_ser_usado, 'inviavel'))
        else:
            print(f'status inesperado em s={s_a_ser_usado:.4f}: {tc}')
            pass
    print("SAIU DO WHILE")
    print(f'Sharpe máximo (diário) ≈ {s_a_ser_usado:.4f}  |  anualizado ≈ {s_a_ser_usado*np.sqrt(252):.2f}')
    # print(historico)
    carteiras_anuais[int(ano_um)] = {
        'pesos':  melhor_pesos,
        'sharpe_anual': s_lo*np.sqrt(252),
    }
    print(f"{ano_um} -> {melhor_pesos}")
    print("+-="*30)
    if 'model' in locals():
        del model
        print('DELETADO DPS DO WHILE')
    else:
        print("Nao consta model")
        pass




=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
Anos totais:  ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-
COMEÇANDO ANO NOVO:  2015
Nao consta model
---------------
Começando Modelo do ano:  2015
Score Atualizado
Retornos atualizados
Excessos Atualizados
Sigmas Atualizados
-----
# ------ CRIAÇÃO DO MODELO
## UTILIZANDO SCORE DO ANO DE:  2015
## UTILIZANDO DADOS DE RETORNO DE:  2016
## UTILIZANDO EXCESSO DO ANO DE:  2016
## UTILIZANDO SIGMAS DO ANO DE:  2016
MODEL.S.VALUE =>  2
Começando o WHILE do ano 2015
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.505
Modelo Resolveu....Condição:  optimal
Valor encontrado para o Sharpe:  0.2575
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.38125
Modelo Resolveu....Condição:  infeasible
Atualização do S_HI : 0.31937499999999

In [14]:
carteiras_anuais

{2016: {'pesos': {0: 2.330595929611094e-09,
   1: 1.4828504749610926e-08,
   2: 0.027893063296608792,
   3: 0.0842590620680534,
   4: 0.0,
   5: 2.5290052248789556e-09,
   6: 2.5353464899245465e-09,
   7: 1.604933422914451e-09,
   8: 1.7671317090175754e-09,
   9: 1.805869967250791e-09,
   10: 2.135409831328136e-09,
   11: 1.044514517499919e-08,
   12: 0.0538779389185768,
   13: 1.0540503742302939e-08,
   14: 0.0,
   15: 1.1298739750092706e-09,
   16: 1.8653155174888056e-09,
   17: 0.0,
   18: 2.599076811468793e-09,
   19: 2.293236430362913e-09,
   20: 0.13987659656817758,
   21: 7.056931940282786e-10,
   22: 0.0,
   23: 3.2308594194120865e-08,
   24: 1.7628785560777982e-09,
   25: 1.2561825006087339e-09,
   26: 2.415505088377872e-09,
   27: 1.7098859019916296e-09,
   28: 1.730714678245653e-09,
   29: 0.0,
   30: 0.17423100710149386,
   31: 8.649042763889123e-09,
   32: 3.0375844946613965e-09,
   33: 0.17485602595694588,
   34: 1.20515903192222e-09,
   35: 1.4665188066459006e-09,
   36:

In [15]:
linhas = []
for an in anos:
    an = int(an.split('-')[0])+1
    print(an)
    for k, v in carteiras_anuais[an]['pesos'].items():
        if v >= vb_peso_minimo:
            linhas.append({'ano': an, 'ativo': lista_ativos_finais[k], 'peso': round(v, 4)})

df_portfolios = pd.DataFrame(linhas)

# # visões instantâneas:
# df_portfolios[df_portfolios['ano'] == 2016].sort_values('peso', ascending=False)  # uma carteira
# df_portfolios.pivot(index='ativo', columns='ano', values='peso')                  # matriz ativo × ano
# df_portfolios.groupby('ativo')['ano'].count().sort_values(ascending=False)  


2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


In [16]:
df_portfolios

,ano,ativo,peso
0,2016,ANIM3,0.0279
1,2016,AXIA3,0.0843
2,2016,BRAP4,0.0539
3,2016,CSMG3,0.1399
4,2016,EQTL3,0.1742
...,...,...,...
102,2026,RADL3,0.1423
103,2026,SLCE3,0.0338
104,2026,USIM5,0.0294
105,2026,WEGE3,0.1545


In [17]:
df_portfolios.to_csv('carteiras_piotroski.csv')
